In [1]:
import os
os.environ["R_MAX_VSIZE"] = "100Gb"  # Increase R's maximum vector size if needed

# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
import numpy as np
import harmonypy as hm
import scanorama
import gffutils
from collections import defaultdict

# sys.path.append('../utils')
# from functions import * 

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

sns.set(style="whitegrid")

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis
scanpy==1.10.4 anndata==0.11.3 umap==0.5.6 numpy==1.26.4 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.6.1 statsmodels==0.14.2 igraph==0.11.5 louvain==0.8.2 pynndescent==0.5.12


In [2]:
# Load gene gtf file for gene length 
gtf_hg38 = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode.v45.primary_assembly.annotation.gtf"
db_file = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode_hg38.db"

def extract_gene_transcript_info(gtf_file, db_file):
    """
    Parses a GENCODE GTF file to compute:
    - Mean transcript length per gene
    - Number of transcripts per gene
    - Gene biotype (protein-coding, lncRNA, etc.)
    - Vector of transcript types per gene
    - Gene name
    
    Returns a DataFrame with gene_id, gene_name, mean_transcript_length, num_transcripts, 
    main_biotype, and transcript_biotypes.
    """
    
    # Check if the database file already exists 
    if os.path.exists(db_file):
        print("Using existing GTF database.")
        db = gffutils.FeatureDB(db_file, keep_order=True)
        print("Database loaded successfully!")
    else:
        print("Creating GTF database (this may take a few minutes)...")
        db = gffutils.create_db(gtf_file, db_file, force=True, keep_order=True, disable_infer_transcripts=False, disable_infer_genes=True)
        print("Database created successfully!")
    
    gene_transcript_lengths = defaultdict(list)
    gene_names = {}
    gene_biotypes = defaultdict(set)  # To collect transcript biotypes
    transcript_counts = defaultdict(int)

    # Iterate over all transcripts in the GTF file
    print("Processing transcripts to compute annotations...")
    for transcript in tqdm(db.features_of_type("transcript"), desc="Processing Transcripts", unit=" transcript"):
        gene_id = transcript.attributes["gene_id"][0]
        gene_name = transcript.attributes.get("gene_name", ["unknown"])[0]
        transcript_biotype = transcript.attributes.get("transcript_type", ["unknown"])[0]  # Extract transcript biotype

        # Compute transcript length (sum of exon lengths)
        transcript_length = sum(exon.end - exon.start + 1 for exon in db.children(transcript, featuretype="exon"))

        gene_transcript_lengths[gene_id].append(transcript_length)
        gene_names[gene_id] = gene_name
        gene_biotypes[gene_id].add(transcript_biotype)  # Store unique transcript types
        transcript_counts[gene_id] += 1  # Count number of transcripts per gene

    print("Finished processing transcripts.")

    # Compute mean transcript length per gene
    gene_mean_lengths = {gene: sum(lengths) / len(lengths) for gene, lengths in gene_transcript_lengths.items()}

    # Convert to DataFrame
    gene_info_df = pd.DataFrame(
        {"gene_id": list(gene_mean_lengths.keys()), 
         "gene_name": [gene_names[g] for g in gene_mean_lengths.keys()], 
         "mean_transcript_length": list(gene_mean_lengths.values()),
         "num_transcripts": [transcript_counts[g] for g in gene_mean_lengths.keys()],
         "transcript_biotypes": [", ".join(sorted(gene_biotypes[g])) for g in gene_mean_lengths.keys()]  # All biotypes
        }
    )

    return gene_info_df

# Run the function
gene_info_df = extract_gene_transcript_info(gtf_hg38, db_file)
gene_info_df

Using existing GTF database.
Database loaded successfully!
Processing transcripts to compute annotations...


Processing Transcripts: 1990 transcript [00:45, 75.83 transcript/s] 

In [ ]:
# follow recommended practices for smart-seq2 raw count normalization
# https://docs.scvi-tools.org/en/1.0.0/tutorials/notebooks/tabula_muris.html

In [ ]:
# set up paths for files 
gene_exp = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/matrix.csv" #intronic and exonic reads 
ab_adata = sc.read_csv(gene_exp)
print(f"Done reading {gene_exp}")

# read in metadata
metadata = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/INFO/metadata.csv"
metadata = pd.read_csv(metadata)

WD="/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression"
today = datetime.datetime.now()
today = today.strftime("%Y-%m-%d")
ab_adata.var['gene_name'] = ab_adata.var_names
ab_adata.obs["dataset"] = "allen_brain"

# Ensure metadata matches the cells in adata
if 'sample_name' in metadata.columns:
    metadata.set_index('sample_name', inplace=True)
    ab_adata.obs = metadata.loc[ab_adata.obs_names]  # Align metadata to the cells in adata
else:
    raise ValueError("Metadata must have a column named 'sample_name' to match with gene expression data.")

# let's keep just the non-neuronal cells from ab_data 
ab_adata = ab_adata[ab_adata.obs["class_label"] == "Non-neuronal"].copy()
ab_adata.var.rename(columns={"gene_name":"gene_symbol"}, inplace=True)
ab_adata.layers["raw_counts"] = ab_adata.X.copy()

# load in tabula sapien data 
tabsap_adata = sc.read_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TS_figshare/TabulaSapiens.h5ad")
tabsap_adata = tabsap_adata[tabsap_adata.obs["method"]=="smartseq2"].copy()
tabsap_adata.obs["dataset"] = "tabula_sapiens"
print("Number of genes in common between the two datasets:", len(set(tabsap_adata.var["gene_symbol"]).intersection(set(ab_adata.var["gene_symbol"]))))

In [ ]:
# Add gene length information to the adata object 
ab_adata.var, tabsap_adata.var

In [ ]:
# I want to only color Microglia and Macrophage cells and monocyte 
cells_color = ["Microglia", "macrophage", "monocyte"]

combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined"].apply(
    lambda x: x if x in cells_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["source", "cell_type_combined_simplified"], palette="tab20",
    title="Combined Datasets UMAP (post Harmony!)",
)


In [ ]:
combined.obs.source.value_counts()

In [ ]:
combined.obs.subclass_label.value_counts()

In [ ]:
silhouette_batch = silhouette_score(combined.obsm["X_umap"], combined.obs["dataset"])
silhouette_biology = silhouette_score(combined.obsm["X_umap"], combined.obs["cell_type_combined_simplified"])
print(f"Silhouette Score by Batch: {silhouette_batch}")
print(f"Silhouette Score by Cell Type: {silhouette_biology}")

In [ ]:
sce.pp.harmony_integrate(combined, 'source')
# Check that Harmony has updated the PCA representation
print("Keys in obsm after Harmony:", combined.obsm.keys())

In [ ]:
# Compute neighbors and UMAP using the Harmony-corrected PCA
sc.pp.neighbors(combined, use_rep="X_pca_harmony")
sc.tl.umap(combined)

In [ ]:
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["source", "cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (post Harmony!)",
)


In [ ]:
# Get cell type counts and identify cell types with more than 1000 cells
cell_type_counts = combined.obs["cell_type_combined"].value_counts()
cell_types_to_color = cell_type_counts[cell_type_counts >= 500].index.tolist()

# Add a new "simplified" column where infrequent cell types are labeled as "Other"
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined"].apply(
    lambda x: x if x in cell_types_to_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")

# Define colors: assign distinct colors to frequent cell types, and use grey for "Other"
cell_type_colors = {
    cell_type: color for cell_type, color in zip(cell_types_to_color, sns.color_palette("husl", len(cell_types_to_color)))
}
cell_type_colors["Other"] = "grey"

# Update `uns` with the colors for visualization
combined.uns["cell_type_combined_simplified_colors"] = [
    cell_type_colors.get(ct, "grey") for ct in combined.obs["cell_type_combined_simplified"].cat.categories
]

# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (Simplified)",
)


In [ ]:
# Define a function to clean up cell type labels
def clean_cell_type(cell_type):
    if "endothelial" in cell_type.lower():
        return "Endothelial"
    elif "b cell" in cell_type.lower():
        return "B cell"
    elif "t cell" in cell_type.lower():
        return "T cell"
    elif "stem" in cell_type.lower():
        return "Stem cell"
    elif "monocyte" in cell_type.lower():
        return "Monocyte"
    else:
        return cell_type  # Keep the original label if no rule matches
    
combined.obs["clean_cell_type"] = combined.obs["cell_type_combined"].apply(clean_cell_type)


In [ ]:
# Get cell type counts and identify cell types with more than 1000 cells
cell_type_counts = combined.obs["clean_cell_type"].value_counts()
cell_types_to_color = cell_type_counts[cell_type_counts >= 750].index.tolist()

# Add a new "simplified" column where infrequent cell types are labeled as "Other"
combined.obs["cell_type_combined_simplified"] = combined.obs["clean_cell_type"].apply(
    lambda x: x if x in cell_types_to_color else "Other"
)

# Convert the new column to a categorical dtype
combined.obs["cell_type_combined_simplified"] = combined.obs["cell_type_combined_simplified"].astype("category")

# Define colors: assign distinct colors to frequent cell types, and use grey for "Other"
cell_type_colors = {
    cell_type: color for cell_type, color in zip(cell_types_to_color, sns.color_palette("husl", len(cell_types_to_color)))
}
cell_type_colors["Other"] = "grey"

# Update `uns` with the colors for visualization
combined.uns["cell_type_combined_simplified_colors"] = [
    cell_type_colors.get(ct, "grey") for ct in combined.obs["cell_type_combined_simplified"].cat.categories
]

# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["cell_type_combined_simplified"], 
    title="Combined Datasets UMAP (Simplified)",
)


In [ ]:
# Visualize with UMAP
sc.pl.umap(
    combined, 
    color=["organ_tissue"])

In [ ]:
combined[combined.obs["cell_type_combined_simplified"] == "macrophage"]

In [ ]:
# plot only macrophage cell type on umap
sc.pl.umap(combined[combined.obs["cell_type_combined_simplified"] == "macrophage"], color="organ_tissue")

In [ ]:
combined.obs.organ_tissue.value_counts()

In [ ]:
combined.obs[combined.obs["clean_cell_type"] == "T cell"].dataset.value_counts()

In [ ]:
combined.obs[combined.obs["clean_cell_type"] == "macrophage"].dataset.value_counts()

In [ ]:
# save combined data 
!pwd

In [ ]:
combined.var

In [ ]:
if "cell_type_combined_simplified_colors" in combined.uns:
    del combined.uns["cell_type_combined_simplified_colors"]

In [ ]:
today = datetime.date.today()
today = today.strftime("%Y-%m-%d")

# save file
combined.write_h5ad(f"/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/TabulaSapien_AllenBrain_combined_nonneuron_adata_{today}.h5ad")